# Phase 3 — Step 10: label-sensitivity, matched-layer LOSO

**Question.** *Does restricting to cells where the depth-bin layer agrees with the metamodel-implied layer change the LOSO headline?* The Phase 3 binaries used `subtype ∈ {5P-IT, 5P-ET}` (full L5 IT/ET pool) and `subtype ∈ {6P-IT, 6P-CT}` (full L6 IT/CT pool) regardless of depth. From the layer × subtype crosstab, 5.6% of the L5 IT/ET pool sits in L4 or L6 depth bins; 9.8% of the L6 IT/CT pool sits in L5. We don't yet know whether those boundary cells are real biology, coregistration noise, or metamodel error (question for the prof). This sensitivity analysis tests the alternative hypothesis directly.

**Protocol — minimal scope per user spec.**

- **Two Phase-1 headline models used uniformly for both tasks**, both as HGB (Phase 1's binding model choice):
  - `Tier G | HGB` — the GKF "within-protocol ceiling" of Phase 1 (PHASE1_GROUND_TRUTH §1, 0.663 GKF on the layer task). Per-neuron tuning fingerprint.
  - `A1+B+C1+D1 | HGB` — the long-row headline of Phase 1 (stage 4, 0.639 long-row HGB).
- We deliberately *do not* use the per-task winners from notebooks 03 / 04 v2 — picking the model that won on the same task whose population we're now perturbing is a circular test (model selection and population selection get entangled). The headline models are fixed a priori, so the matched-vs-full Δ is purely about the population. Running both Phase-1 headlines is itself a robustness check: if Δ matched − full is consistent across the two headlines, we trust the conclusion; if they disagree, that disagreement is part of the result.
- LOSO only (no GKF); `cc_abs`-residualization included as the SNR control.
- Two populations side-by-side per task:
  - **full**: subtype-only filter (matches the headline of notebooks 03/04 v2).
  - **matched**: subtype filter **AND** depth bin matches the metamodel-implied layer.
- LOSO validity rule re-evaluated on each population (the matched population may lose scans that no longer have minority ≥ 5).

**Three readings of the result, prespecified before running:**

| matched vs full | reading |
|---|---|
| matched ≈ full (Δ < 0.03) | boundary cells are not driving the result; both populations agree. The full-pool headline stands. |
| matched > full (Δ ≥ +0.03) | boundary cells were hurting (the metamodel may be over-extending into wrong depth bins). The depth-restricted version is the cleaner signal. |
| matched < full (Δ ≤ −0.03) | boundary cells were carrying real signal; restricting to depth-matched cells drops informative data. The full-pool result is the one to keep. |

We don't pick the threshold post-hoc; ±0.03 is the planning-doc convention for "noise floor of LOSO mean across 6–10 valid scans".


## 1. Setup

In [1]:
from __future__ import annotations

import sys, time, json, warnings
from pathlib import Path

import numpy as np
import pandas as pd

REPO_ROOT = Path('..').resolve().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler
from sklearn.pipeline import Pipeline
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings('ignore', category=ConvergenceWarning)
warnings.filterwarnings('ignore', category=FutureWarning, module='sklearn')

from src.config import (PROCESSED_TABLES_DIR, PROCESSED_FEATURES_DIR,
                        PROCESSED_RESULTS_DIR, RANDOM_SEED, ensure_dirs)
from src.data.loaders import build_modeling_table
from src.eval.metrics import neuron_level_score, summarize_cv_runs
from src.features.tier_b import B_FEATURE_NAMES
from src.features.tier_c import C1_FEATURE_NAMES
from src.features.tier_d import D_FEATURE_NAMES

ensure_dirs()
np.random.seed(RANDOM_SEED)

DATA_TABLES   = REPO_ROOT / 'data' / 'processed' / 'tables'
DATA_FEATURES = REPO_ROOT / 'data' / 'processed' / 'features'
DATA_RESULTS  = REPO_ROOT / 'data' / 'processed' / 'results'

L5_WINNER_PATH = DATA_RESULTS / 'phase3_l5_it_et_winner.json'
L6_WINNER_PATH = DATA_RESULTS / 'phase3_l6_it_ct_winner.json'
SENS_OUT       = DATA_RESULTS / 'phase3_label_sensitivity.parquet'
SENS_PERSCAN_OUT = DATA_RESULTS / 'phase3_label_sensitivity_perscan.parquet'

LOSO_MIN_MINORITY = 5  # same rule as Step 1


## 2. Build the long-row modelling table (same join chain as 03/04 v2)

In [2]:
# Same join chain used by 02/03/04 v2.
X_a1, y_a1, g_a1, f_a1, df_a1 = build_modeling_table(
    level='A1', blocks=['amp','shape'], label='celltype_label')

b_hash = pd.read_parquet(DATA_FEATURES / 'B_per_hash.parquet',
    columns=['nucleus_id','condition_hash'] + list(B_FEATURE_NAMES))
df_a1b = df_a1.merge(b_hash, on=['nucleus_id','condition_hash'],
                     how='inner', validate='one_to_one')

c1 = pd.read_parquet(DATA_FEATURES / 'C1_per_hash.parquet',
    columns=['nucleus_id','condition_hash'] + list(C1_FEATURE_NAMES))
df_a1bc1 = df_a1b.merge(c1, on=['nucleus_id','condition_hash'],
                        how='left', validate='one_to_one')

d = pd.read_parquet(DATA_FEATURES / 'D_per_hash.parquet',
    columns=['condition_hash'] + list(D_FEATURE_NAMES))
df_full = df_a1bc1.merge(d, on='condition_hash', how='left', validate='many_to_one')

G = pd.read_parquet(DATA_FEATURES / 'G_per_neuron.parquet')
g_cols = [c for c in G.columns if c.startswith('g_')]
df_long = df_full.merge(G[['nucleus_id'] + g_cols], on='nucleus_id',
                        how='left', validate='many_to_one')

# Layer-bin column: from units_working via build_modeling_table's df.
# build_modeling_table joined units_working columns (cc_abs, pt_position_y,
# session_key) but we also need 'layer_label'. Pull it explicitly.
units = pd.read_parquet(DATA_TABLES / 'units_working.parquet',
                        columns=['nucleus_id','layer_label'])
df_long = df_long.merge(units, on='nucleus_id', how='left', validate='many_to_one')

print(f'long-row table: {df_long.shape}')
print(f'  unique neurons: {df_long["nucleus_id"].nunique():,}')
print(f'  unique scans  : {df_long["session_key"].nunique()}')
print()

# Quick sanity: layer × subtype on neurons (to motivate the matched filter)
nd = df_long.drop_duplicates('nucleus_id')
print('neurons by layer × celltype:')
print(pd.crosstab(nd['layer_label'], nd['celltype_label']).to_string())


long-row table: (1209720, 169)
  unique neurons: 8,895
  unique scans  : 13

neurons by layer × celltype:
celltype_label   23P    4P  5P-ET  5P-IT  5P-NP  6P-CT  6P-IT
layer_label                                                  
L2/3            4160    87      0      0      0      0      0
L4                38  2600      4     28      0      0      0
L5                 0   158    289   1117     18     13     20
L6                 0     0     27     24      8    108    196


## 3. Headline models — fixed a priori, NOT the per-task winners

We run **two** Phase-1 headline candidates, both as HGB (Phase 1's binding model choice), uniformly for both L5 IT/ET and L6 IT/CT:

- `Tier G | HGB` — Phase 1 GKF ceiling (0.663 on the layer task).
- `A1+B+C1+D1 | HGB` — Phase 1 long-row headline (stage 4, 0.639).

Running both lets us check that the matched-vs-full Δ is a property of the population, not of the headline-model choice.


In [3]:
# === HEADLINE MODELS (fixed a priori, NOT the per-task winners) ===
HEADLINES = [
    {'name': 'TierG_HGB',       'block': 'G',          'model': 'HGB',
     'description': 'Phase 1 GKF ceiling (PHASE1_GROUND_TRUTH §1)'},
    {'name': 'A1BC1D1_HGB',     'block': 'A1+B+C1+D1', 'model': 'HGB',
     'description': 'Phase 1 long-row headline (stage 4)'},
]
print('Headline models (used for both tasks, fixed a priori):')
for h in HEADLINES:
    print(f"  {h['name']:<14s}  block={h['block']:<11s}  model={h['model']:<6s}  ({h['description']})")
print()

# Per-task winners — context only, not used for the sensitivity LOSO.
def load_winner_meta(path: Path):
    if path.exists():
        with open(path) as f:
            return json.load(f)
    return None

l5_meta = load_winner_meta(L5_WINNER_PATH)
l6_meta = load_winner_meta(L6_WINNER_PATH)
if l5_meta:
    print(f'(For context — L5 IT/ET per-task LOSO winner: {l5_meta.get("winner_block")} | '
          f'{l5_meta.get("winner_model")}; not used here.)')
if l6_meta:
    print(f'(For context — L6 IT/CT per-task LOSO winner: {l6_meta.get("winner_block")} | '
          f'{l6_meta.get("winner_model")}; not used here.)')


Headline models (used for both tasks, fixed a priori):
  TierG_HGB       block=G            model=HGB     (Phase 1 GKF ceiling (PHASE1_GROUND_TRUTH §1))
  A1BC1D1_HGB     block=A1+B+C1+D1   model=HGB     (Phase 1 long-row headline (stage 4))

(For context — L5 IT/ET per-task LOSO winner: A1+B+C1+D1 | HGB; not used here.)
(For context — L6 IT/CT per-task LOSO winner: A1+B+C1+D1 | LogReg; not used here.)


## 4. Helpers

In [4]:
amp_cols   = [c for c in df_long.columns if c.startswith('amp_')]
shape_cols = [c for c in df_long.columns if c.startswith('shape_')]
a1_cols    = amp_cols + shape_cols
b_cols     = list(B_FEATURE_NAMES)
c1_cols    = list(C1_FEATURE_NAMES)
d_cols     = list(D_FEATURE_NAMES)

BLOCKS = {
    'G':            list(g_cols),
    'A1+B':         a1_cols + b_cols,
    'A1+B+C1':      a1_cols + b_cols + c1_cols,
    'A1+B+C1+D1':   a1_cols + b_cols + c1_cols + d_cols,
    'G+B+C1':       list(g_cols) + b_cols + c1_cols,
}


def make_pipeline(model_name: str) -> Pipeline:
    if model_name == 'LogReg':
        return Pipeline([
            ('impute', SimpleImputer(strategy='median')),
            ('scale',  RobustScaler()),
            ('clf',    LogisticRegression(
                penalty='l2', C=1.0, solver='lbfgs', max_iter=400,
                class_weight='balanced', n_jobs=-1, random_state=RANDOM_SEED)),
        ])
    if model_name == 'HGB':
        return Pipeline([
            ('clf', HistGradientBoostingClassifier(
                max_iter=100, max_depth=8, learning_rate=0.05,
                l2_regularization=1.0,
                early_stopping=True, n_iter_no_change=10,
                random_state=RANDOM_SEED)),
        ])
    raise ValueError(model_name)


def _residualize(X_tr, X_te, c_tr, c_te):
    med = np.nanmedian(c_tr)
    c_tr_f = np.where(np.isnan(c_tr), med, c_tr)
    c_te_f = np.where(np.isnan(c_te), med, c_te)
    Xt = X_tr.copy(); Xe = X_te.copy()
    for j in range(X_tr.shape[1]):
        f_tr = Xt[:, j]; m = ~np.isnan(f_tr)
        if m.sum() < 5: continue
        c_fit = c_tr_f[m, 0]; f_fit = f_tr[m]
        cm_, fm_ = c_fit.mean(), f_fit.mean()
        denom = ((c_fit - cm_) ** 2).sum()
        if denom < 1e-12: continue
        b = ((c_fit - cm_) * (f_fit - fm_)).sum() / denom
        a = fm_ - b * cm_
        Xt[:, j] = f_tr - (a + b * c_tr_f[:, 0])
        Xe[:, j] = Xe[:, j] - (a + b * c_te_f[:, 0])
    return Xt, Xe


def loso_run_long(df_pop, feature_cols, model_name, classes,
                  residualize=False, min_minority=LOSO_MIN_MINORITY):
    y    = df_pop['celltype_label'].to_numpy()
    g    = df_pop['nucleus_id'].to_numpy()
    sess = df_pop['session_key'].to_numpy()
    cc   = df_pop['cc_abs'].to_numpy(np.float64)
    X    = df_pop[feature_cols].to_numpy(np.float64)

    rows = []
    for sk in sorted(np.unique(sess).tolist()):
        te = sess == sk; tr = ~te
        if tr.sum() == 0 or te.sum() == 0: continue
        if len(np.unique(y[tr])) < 2: continue
        if residualize:
            Xt, Xe = _residualize(X[tr], X[te], cc[tr].reshape(-1,1), cc[te].reshape(-1,1))
        else:
            Xt, Xe = X[tr], X[te]
        pipe = make_pipeline(model_name)
        sw = compute_sample_weight('balanced', y[tr])
        pipe.fit(Xt, y[tr], clf__sample_weight=sw)
        proba = pipe.predict_proba(Xe)
        sc = neuron_level_score(y[te], proba, g[te], pipe.classes_)
        # per-scan validity
        n_classes_present = (pd.Series(sc['y_neuron_true']).value_counts() >= min_minority).sum()
        # binary-style validity: minority class >= min_minority AND both present
        nA = int((sc['y_neuron_true'] == classes[0]).sum())
        nB = int((sc['y_neuron_true'] == classes[1]).sum())
        valid = (nA > 0) and (nB > 0) and (min(nA, nB) >= min_minority)
        rows.append({
            'held_out_scan': sk,
            'n_neurons_test': sc['n_neurons'],
            f'n_test_{classes[0]}': nA, f'n_test_{classes[1]}': nB,
            'minority_n': min(nA, nB),
            'both_classes': (nA > 0 and nB > 0),
            'valid_for_loso': bool(valid),
            'balanced_accuracy': sc['balanced_accuracy'],
            'macro_f1':          sc['macro_f1'],
            f'recall_{classes[0]}': sc['per_class_recall'].get(classes[0], float('nan')),
            f'recall_{classes[1]}': sc['per_class_recall'].get(classes[1], float('nan')),
            'cm':                sc['confusion_matrix'].tolist(),
        })
    return rows


def aggregate(rows):
    df_ps = pd.DataFrame(rows)
    out = {}
    for tag, sub in (('all', df_ps), ('valid', df_ps[df_ps['valid_for_loso']])):
        v = sub['balanced_accuracy'].dropna().to_numpy(float)
        out[f'bal_acc_{tag}_mean'] = float(v.mean()) if len(v) else float('nan')
        out[f'bal_acc_{tag}_std']  = float(v.std())  if len(v) else float('nan')
        out[f'n_scans_{tag}']      = int(len(v))
    return out


print('Helpers ready.')


Helpers ready.


## 5. L5 IT/ET sensitivity — full vs matched (depth bin == L5)

The headline of notebook 03 v2 was the *full* L5 IT/ET pool: 1,489 neurons (1,169 IT + 320 ET), regardless of depth bin. The *matched* population restricts to those with depth bin `L5`: 1,406 neurons (1,117 IT + 289 ET) — drops 5.6% (52 IT cells in L4/L6 depth bins, 31 ET cells in L4/L6 depth bins).


In [5]:
# Full L5 IT/ET subset
df_l5_full = df_long[df_long['celltype_label'].isin(['5P-IT','5P-ET'])].copy()
nd_full = df_l5_full.drop_duplicates('nucleus_id')

# Matched: subtype + layer == L5
df_l5_matched = df_l5_full[df_l5_full['layer_label'] == 'L5'].copy()
nd_matched = df_l5_matched.drop_duplicates('nucleus_id')

print(f'L5 IT/ET FULL    : long rows {len(df_l5_full):,}, '
      f'neurons {len(nd_full)} (5P-IT={int((nd_full["celltype_label"]=="5P-IT").sum())}, '
      f'5P-ET={int((nd_full["celltype_label"]=="5P-ET").sum())}), '
      f'scans {df_l5_full["session_key"].nunique()}')
print(f'L5 IT/ET MATCHED : long rows {len(df_l5_matched):,}, '
      f'neurons {len(nd_matched)} (5P-IT={int((nd_matched["celltype_label"]=="5P-IT").sum())}, '
      f'5P-ET={int((nd_matched["celltype_label"]=="5P-ET").sum())}), '
      f'scans {df_l5_matched["session_key"].nunique()}')

# Per-scan IT/ET counts for each (re-evaluate validity on the matched population)
print()
print('per-scan counts (FULL):')
print(pd.crosstab(nd_full['session_key'], nd_full['celltype_label']).to_string())
print()
print('per-scan counts (MATCHED):')
print(pd.crosstab(nd_matched['session_key'], nd_matched['celltype_label']).to_string())


L5 IT/ET FULL    : long rows 202,504, neurons 1489 (5P-IT=1169, 5P-ET=320), scans 11
L5 IT/ET MATCHED : long rows 191,216, neurons 1406 (5P-IT=1117, 5P-ET=289), scans 10

per-scan counts (FULL):
celltype_label  5P-ET  5P-IT
session_key                 
4_7                66     39
5_6                44    146
5_7                16    134
6_2                 9    150
6_4                 5    206
6_6                16    158
6_7                33    183
7_3                64     92
7_5                10      8
8_5                56     53
9_3                 1      0

per-scan counts (MATCHED):
celltype_label  5P-ET  5P-IT
session_key                 
4_7                66     32
5_6                44    145
5_7                15    131
6_2                 8    144
6_4                 0    200
6_6                 4    148
6_7                24    167
7_3                64     92
7_5                 9      8
8_5                55     50


In [6]:
L5_CLASSES = ('5P-IT', '5P-ET')
results = []
perscan_all = []

for h in HEADLINES:
    block = h['block']; model = h['model']; hname = h['name']
    print(f'\n>>> L5 IT/ET headline: {hname} ({block} | {model})')
    for tag, df_pop in [('full', df_l5_full), ('matched', df_l5_matched)]:
        for resid in (False, True):
            rows = loso_run_long(df_pop, BLOCKS[block], model,
                                 classes=L5_CLASSES, residualize=resid)
            agg = aggregate(rows)
            for r in rows:
                perscan_all.append({'task':'L5_IT_vs_ET','headline':hname,
                                    'population':tag,'residualized':resid, **r})
            results.append({
                'task':'L5_IT_vs_ET','headline': hname,
                'population': tag,
                'block': block, 'model': model,
                'residualized': resid,
                'n_neurons': int(df_pop['nucleus_id'].nunique()),
                **agg,
            })
            label = 'cc_abs-resid' if resid else 'raw'
            print(f'  [L5 {tag:<7s} | {label:<13s}] valid({agg["n_scans_valid"]}) bal_acc = '
                  f'{agg["bal_acc_valid_mean"]:.3f} ± {agg["bal_acc_valid_std"]:.3f}')

# L5 summary by headline
print('\n=== L5 IT/ET — FULL vs MATCHED LOSO, by headline ===')
ldf = pd.DataFrame([r for r in results if r['task']=='L5_IT_vs_ET'])
ldf['summary_str'] = ldf.apply(
    lambda r: f"{r['bal_acc_valid_mean']:.3f} ± {r['bal_acc_valid_std']:.3f} (n={int(r['n_scans_valid'])})", axis=1)
print(ldf[['headline','population','residualized','n_neurons','summary_str']].to_string(index=False))

# Δ matched − full per headline
def deltaL5(headline, resid):
    full_v = ldf[(ldf['headline']==headline) & (ldf['population']=='full') & (ldf['residualized']==resid)]['bal_acc_valid_mean'].iloc[0]
    match_v = ldf[(ldf['headline']==headline) & (ldf['population']=='matched') & (ldf['residualized']==resid)]['bal_acc_valid_mean'].iloc[0]
    return full_v, match_v, match_v - full_v

print('\n=== Δ matched − full (L5 IT/ET) ===')
for h in HEADLINES:
    for resid in (False, True):
        f, m, d = deltaL5(h['name'], resid)
        prep = 'cc_abs-resid' if resid else 'raw         '
        print(f"  {h['name']:<14s} | {prep} : full={f:.3f}, matched={m:.3f}, Δ={d:+.3f}")



>>> L5 IT/ET headline: TierG_HGB (G | HGB)


/opt/anaconda3/envs/neuroscience/lib/python3.10/site-packages/sklearn/metrics/_classification.py:2801: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


  [L5 full    | raw          ] valid(10) bal_acc = 0.527 ± 0.035


/opt/anaconda3/envs/neuroscience/lib/python3.10/site-packages/sklearn/metrics/_classification.py:2801: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


  [L5 full    | cc_abs-resid ] valid(10) bal_acc = 0.548 ± 0.077


...

  [L5 full    | cc_abs-resid ] valid(10) bal_acc = 0.582 ± 0.079


/opt/anaconda3/envs/neuroscience/lib/python3.10/site-packages/sklearn/metrics/_classification.py:2801: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


  [L5 matched | raw          ] valid(8) bal_acc = 0.598 ± 0.096


/opt/anaconda3/envs/neuroscience/lib/python3.10/site-packages/sklearn/metrics/_classification.py:2801: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


  [L5 matched | cc_abs-resid ] valid(8) bal_acc = 0.556 ± 0.118

=== L5 IT/ET — FULL vs MATCHED LOSO, by headline ===
   headline population  residualized  n_neurons          summary_str
  TierG_HGB       full         False       1489 0.527 ± 0.035 (n=10)
  TierG_HGB       full          True       1489 0.548 ± 0.077 (n=10)
  TierG_HGB    matched         False       1406  0.532 ± 0.033 (n=8)
  TierG_HGB    matched          True       1406  0.521 ± 0.034 (n=8)
A1BC1D1_HGB       full         False       1489 0.581 ± 0.094 (n=10)
A1BC1D1_HGB       full          True       1489 0.582 ± 0.079 (n=10)
A1BC1D1_HGB    matched         False       1406  0.598 ± 0.096 (n=8)
A1BC1D1_HGB    matched          True       1406  0.556 ± 0.118 (n=8)

=== Δ matched − full (L5 IT/ET) ===
  TierG_HGB      | raw          : full=0.527, matched=0.532, Δ=+0.004
  TierG_HGB      | cc_abs-resid : full=0.548, matched=0.521, Δ=-0.027
  A1BC1D1_HGB    | raw          : full=0.581, matched=0.598, Δ=+0.017
  A1BC1D1_HGB 

## 6. L6 IT/CT sensitivity — full vs matched (depth bin == L6)

In [7]:
df_l6_full = df_long[df_long['celltype_label'].isin(['6P-IT','6P-CT'])].copy()
nd_full6 = df_l6_full.drop_duplicates('nucleus_id')
df_l6_matched = df_l6_full[df_l6_full['layer_label'] == 'L6'].copy()
nd_matched6 = df_l6_matched.drop_duplicates('nucleus_id')

print(f'L6 IT/CT FULL    : long rows {len(df_l6_full):,}, '
      f'neurons {len(nd_full6)} (6P-IT={int((nd_full6["celltype_label"]=="6P-IT").sum())}, '
      f'6P-CT={int((nd_full6["celltype_label"]=="6P-CT").sum())}), '
      f'scans {df_l6_full["session_key"].nunique()}')
print(f'L6 IT/CT MATCHED : long rows {len(df_l6_matched):,}, '
      f'neurons {len(nd_matched6)} (6P-IT={int((nd_matched6["celltype_label"]=="6P-IT").sum())}, '
      f'6P-CT={int((nd_matched6["celltype_label"]=="6P-CT").sum())}), '
      f'scans {df_l6_matched["session_key"].nunique()}')
print()
print('per-scan counts (FULL):')
print(pd.crosstab(nd_full6['session_key'], nd_full6['celltype_label']).to_string())
print()
print('per-scan counts (MATCHED):')
print(pd.crosstab(nd_matched6['session_key'], nd_matched6['celltype_label']).to_string())


L6 IT/CT FULL    : long rows 45,832, neurons 337 (6P-IT=216, 6P-CT=121), scans 7
L6 IT/CT MATCHED : long rows 41,344, neurons 304 (6P-IT=196, 6P-CT=108), scans 6

per-scan counts (FULL):
celltype_label  6P-CT  6P-IT
session_key                 
4_7                 1      0
5_6                50     46
5_7                24     57
6_2                 9     27
6_4                14     41
6_6                11     23
6_7                12     22

per-scan counts (MATCHED):
celltype_label  6P-CT  6P-IT
session_key                 
5_6                50     46
5_7                24     57
6_2                 8     25
6_4                12     38
6_6                10     19
6_7                 4     11


In [8]:
L6_CLASSES = ('6P-IT', '6P-CT')
for h in HEADLINES:
    block = h['block']; model = h['model']; hname = h['name']
    print(f'\n>>> L6 IT/CT headline: {hname} ({block} | {model})')
    for tag, df_pop in [('full', df_l6_full), ('matched', df_l6_matched)]:
        for resid in (False, True):
            rows = loso_run_long(df_pop, BLOCKS[block], model,
                                 classes=L6_CLASSES, residualize=resid)
            agg = aggregate(rows)
            for r in rows:
                perscan_all.append({'task':'L6_IT_vs_CT','headline':hname,
                                    'population':tag,'residualized':resid, **r})
            results.append({
                'task':'L6_IT_vs_CT','headline': hname,
                'population': tag,
                'block': block, 'model': model,
                'residualized': resid,
                'n_neurons': int(df_pop['nucleus_id'].nunique()),
                **agg,
            })
            label = 'cc_abs-resid' if resid else 'raw'
            print(f'  [L6 {tag:<7s} | {label:<13s}] valid({agg["n_scans_valid"]}) bal_acc = '
                  f'{agg["bal_acc_valid_mean"]:.3f} ± {agg["bal_acc_valid_std"]:.3f}')

print('\n=== L6 IT/CT — FULL vs MATCHED LOSO, by headline ===')
l6df = pd.DataFrame([r for r in results if r['task']=='L6_IT_vs_CT'])
l6df['summary_str'] = l6df.apply(
    lambda r: f"{r['bal_acc_valid_mean']:.3f} ± {r['bal_acc_valid_std']:.3f} (n={int(r['n_scans_valid'])})", axis=1)
print(l6df[['headline','population','residualized','n_neurons','summary_str']].to_string(index=False))

def deltaL6(headline, resid):
    full_v = l6df[(l6df['headline']==headline) & (l6df['population']=='full') & (l6df['residualized']==resid)]['bal_acc_valid_mean'].iloc[0]
    match_v = l6df[(l6df['headline']==headline) & (l6df['population']=='matched') & (l6df['residualized']==resid)]['bal_acc_valid_mean'].iloc[0]
    return full_v, match_v, match_v - full_v

print('\n=== Δ matched − full (L6 IT/CT) ===')
for h in HEADLINES:
    for resid in (False, True):
        f, m, d = deltaL6(h['name'], resid)
        prep = 'cc_abs-resid' if resid else 'raw         '
        print(f"  {h['name']:<14s} | {prep} : full={f:.3f}, matched={m:.3f}, Δ={d:+.3f}")



>>> L6 IT/CT headline: TierG_HGB (G | HGB)


/opt/anaconda3/envs/neuroscience/lib/python3.10/site-packages/sklearn/metrics/_classification.py:2801: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


  [L6 full    | raw          ] valid(6) bal_acc = 0.522 ± 0.032


/opt/anaconda3/envs/neuroscience/lib/python3.10/site-packages/sklearn/metrics/_classification.py:2801: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


  [L6 full    | cc_abs-resid ] valid(6) bal_acc = 0.512 ± 0.035
  [L6 matched | raw          ] valid(5) bal_acc = 0.508 ± 0.046
  [L6 matched | cc_abs-resid ] valid(5) bal_acc = 0.524 ± 0.043

>>> L6 IT/CT headline: A1BC1D1_HGB (A1+B+C1+D1 | HGB)


/opt/anaconda3/envs/neuroscience/lib/python3.10/site-packages/sklearn/metrics/_classification.py:2801: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


  [L6 full    | raw          ] valid(6) bal_acc = 0.569 ± 0.058


/opt/anaconda3/envs/neuroscience/lib/python3.10/site-packages/sklearn/metrics/_classification.py:2801: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


  [L6 full    | cc_abs-resid ] valid(6) bal_acc = 0.536 ± 0.061
  [L6 matched | raw          ] valid(5) bal_acc = 0.513 ± 0.105
  [L6 matched | cc_abs-resid ] valid(5) bal_acc = 0.549 ± 0.047

=== L6 IT/CT — FULL vs MATCHED LOSO, by headline ===
   headline population  residualized  n_neurons         summary_str
  TierG_HGB       full         False        337 0.522 ± 0.032 (n=6)
  TierG_HGB       full          True        337 0.512 ± 0.035 (n=6)
  TierG_HGB    matched         False        304 0.508 ± 0.046 (n=5)
  TierG_HGB    matched          True        304 0.524 ± 0.043 (n=5)
A1BC1D1_HGB       full         False        337 0.569 ± 0.058 (n=6)
A1BC1D1_HGB       full          True        337 0.536 ± 0.061 (n=6)
A1BC1D1_HGB    matched         False        304 0.513 ± 0.105 (n=5)
A1BC1D1_HGB    matched          True        304 0.549 ± 0.047 (n=5)

=== Δ matched − full (L6 IT/CT) ===
  TierG_HGB      | raw          : full=0.522, matched=0.508, Δ=-0.014
  TierG_HGB      | cc_abs-resid : f

## 7. Combined sensitivity table & save

In [9]:
sens_df = pd.DataFrame(results)
sens_df['summary_str'] = sens_df.apply(
    lambda r: f"{r['bal_acc_valid_mean']:.3f} ± {r['bal_acc_valid_std']:.3f} (n={int(r['n_scans_valid'])})", axis=1)
sens_df['preprocessing'] = sens_df['residualized'].map({True:'cc_abs-resid', False:'raw'})

print('=== PHASE 3 — LABEL SENSITIVITY (matched-layer LOSO) ===')
print(sens_df[['task','headline','population','preprocessing','block','model',
               'n_neurons','summary_str']].to_string(index=False))

# Concise comparison: per task × headline × preprocessing
print('\n=== Δ MATCHED − FULL (per task / per headline) ===')
for task in ('L5_IT_vs_ET','L6_IT_vs_CT'):
    for h in HEADLINES:
        for r in (False, True):
            sub = sens_df[(sens_df['task']==task) & (sens_df['headline']==h['name']) & (sens_df['residualized']==r)]
            full_v   = sub[sub['population']=='full']['bal_acc_valid_mean'].iloc[0]
            match_v  = sub[sub['population']=='matched']['bal_acc_valid_mean'].iloc[0]
            d = match_v - full_v
            prep = 'cc_abs-resid' if r else 'raw         '
            rule = 'matched ≈ full' if abs(d) < 0.03 else ('matched > full (boundary cells hurt)' if d > 0 else 'matched < full (boundary cells helped)')
            print(f"  {task:<13s} | {h['name']:<14s} | {prep} : full={full_v:.3f}, matched={match_v:.3f}, Δ={d:+.3f}  ({rule})")

# Save
sens_df.to_parquet(SENS_OUT, index=False)
print(f'\nwrote {SENS_OUT}  ({SENS_OUT.stat().st_size/1024:.1f} KB, {len(sens_df)} rows)')

ps = pd.DataFrame(perscan_all).copy()
ps['cm'] = ps['cm'].apply(lambda v: json.dumps(v) if v is not None else None)
ps.to_parquet(SENS_PERSCAN_OUT, index=False)
print(f'wrote {SENS_PERSCAN_OUT}  ({SENS_PERSCAN_OUT.stat().st_size/1024:.1f} KB, {len(ps)} rows)')


=== PHASE 3 — LABEL SENSITIVITY (matched-layer LOSO) ===
       task    headline population preprocessing      block model  n_neurons          summary_str
L5_IT_vs_ET   TierG_HGB       full           raw          G   HGB       1489 0.527 ± 0.035 (n=10)
L5_IT_vs_ET   TierG_HGB       full  cc_abs-resid          G   HGB       1489 0.548 ± 0.077 (n=10)
L5_IT_vs_ET   TierG_HGB    matched           raw          G   HGB       1406  0.532 ± 0.033 (n=8)
L5_IT_vs_ET   TierG_HGB    matched  cc_abs-resid          G   HGB       1406  0.521 ± 0.034 (n=8)
L5_IT_vs_ET A1BC1D1_HGB       full           raw A1+B+C1+D1   HGB       1489 0.581 ± 0.094 (n=10)
L5_IT_vs_ET A1BC1D1_HGB       full  cc_abs-resid A1+B+C1+D1   HGB       1489 0.582 ± 0.079 (n=10)
L5_IT_vs_ET A1BC1D1_HGB    matched           raw A1+B+C1+D1   HGB       1406  0.598 ± 0.096 (n=8)
L5_IT_vs_ET A1BC1D1_HGB    matched  cc_abs-resid A1+B+C1+D1   HGB       1406  0.556 ± 0.118 (n=8)
L6_IT_vs_CT   TierG_HGB       full           raw          G  

## 8. Step-10 summary

In [10]:
print('=== PHASE 3 STEP 10 — LABEL SENSITIVITY SUMMARY ===')
print()
print('Headline models (fixed a priori, run on both tasks):')
for h in HEADLINES:
    print(f"  {h['name']:<14s} = {h['block']} | {h['model']}   ({h['description']})")
print()

for task, dfx, getter in [('L5 IT/ET', ldf, deltaL5), ('L6 IT/CT', l6df, deltaL6)]:
    print(f'{task}:')
    for h in HEADLINES:
        for resid in (False, True):
            f, m, d = getter(h['name'], resid)
            prep = 'cc_abs-resid' if resid else 'raw         '
            print(f"  {h['name']:<14s} | {prep} : full={f:.3f}, matched={m:.3f}, Δ={d:+.3f}")
    print()

print('Reading: ±0.03 is the noise floor for LOSO mean across 6–10 valid scans.')
print('  |Δ| < 0.03  → matched ≈ full; boundary cells are not driving the result.')
print('  Δ ≥ +0.03   → matched > full; boundary cells were hurting the decoder.')
print('  Δ ≤ −0.03   → matched < full; boundary cells were carrying real signal.')
print()
print('Robustness: the matched-vs-full conclusion should be CONSISTENT across both')
print('headline models. If the two headlines give qualitatively different Δs, that')
print('disagreement is part of the result and needs interpretation.')


=== PHASE 3 STEP 10 — LABEL SENSITIVITY SUMMARY ===

Headline models (fixed a priori, run on both tasks):
  TierG_HGB      = G | HGB   (Phase 1 GKF ceiling (PHASE1_GROUND_TRUTH §1))
  A1BC1D1_HGB    = A1+B+C1+D1 | HGB   (Phase 1 long-row headline (stage 4))

L5 IT/ET:
  TierG_HGB      | raw          : full=0.527, matched=0.532, Δ=+0.004
  TierG_HGB      | cc_abs-resid : full=0.548, matched=0.521, Δ=-0.027
  A1BC1D1_HGB    | raw          : full=0.581, matched=0.598, Δ=+0.017
  A1BC1D1_HGB    | cc_abs-resid : full=0.582, matched=0.556, Δ=-0.026

L6 IT/CT:
  TierG_HGB      | raw          : full=0.522, matched=0.508, Δ=-0.014
  TierG_HGB      | cc_abs-resid : full=0.512, matched=0.524, Δ=+0.012
  A1BC1D1_HGB    | raw          : full=0.569, matched=0.513, Δ=-0.055
  A1BC1D1_HGB    | cc_abs-resid : full=0.536, matched=0.549, Δ=+0.013

Reading: ±0.03 is the noise floor for LOSO mean across 6–10 valid scans.
  |Δ| < 0.03  → matched ≈ full; boundary cells are not driving the result.
  Δ ≥ +0.03